In [2]:
import os
# Only go up a directory if we are currently in the 'research' folder
if os.getcwd().endswith('research'):
    os.chdir("../")


In [3]:
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

In [4]:
%pwd

'/Users/manish/Desktop/chicken-disease-classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
  root_dir: Path
  source_URL: str
  local_data_file: Path
  unzip_dir: Path

In [6]:
from chicken_disease_classification.constants import *
from chicken_disease_classification.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
  def __init__(
    self,
    config_filepath = CONFIG_FILE_PATH,
    params_filepath = PARAMS_FILE_PATH):

    self.config = read_yaml(config_filepath)
    self.params = read_yaml(params_filepath)

    create_directories([self.config.artifacts_root])

  def get_data_ingestion_config(self) -> DataIngestionConfig:
    config = self.config.data_ingestion 

    create_directories([config.root_dir])

    get_data_ingestion_config = DataIngestionConfig(
      root_dir=config.root_dir,
      source_URL=config.source_URL,
      local_data_file=config.local_data_file,
      unzip_dir=config.unzip_dir
    )

    return get_data_ingestion_config


In [8]:
import os
import urllib.request as request
import zipfile
from chicken_disease_classification import logger
from chicken_disease_classification.utils.common import get_size

In [9]:
class DataIngestion:
  def __init__(self, config: DataIngestionConfig):
    self.config = config

  def download_file(self):
    if not os.path.exists(self.config.local_data_file):
      filename, headers = request.urlretrieve(
        url = self.config.source_URL,
        filename = self.config.local_data_file
      )
      logger.info(f"{filename} download! with following info: \n{headers}")
    else:
      logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

  def extract_zip_file(self):
    """
    Zip_file_path: str
    Extracts the zip file into the data directory
    Function returns None
    """
    unzip_path = self.config.unzip_dir
    os.makedirs = self.config.unzip_dir
    with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
      zip_ref.extractall(unzip_path)

In [10]:
try:
  config = ConfigurationManager()
  data_ingestion_config = config.get_data_ingestion_config()
  data_ingestion = DataIngestion(config=data_ingestion_config)
  data_ingestion.download_file()
  data_ingestion.extract_zip_file()
except Exception as e:
  raise e

[2026-09-27 10:10:30,470: INFO: chicken_disease_classification: yaml file: config/config.yaml loaded successfully]
[2026-09-27 10:10:30,471: INFO: chicken_disease_classification: yaml file: params.yaml loaded successfully]
[2026-09-27 10:10:30,472: INFO: chicken_disease_classification: Created directory at: artifacts]
[2026-09-27 10:10:30,472: INFO: chicken_disease_classification: Created directory at: artifacts/data_ingestion]
[2026-09-27 10:10:32,456: INFO: chicken_disease_classification: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: A532:3B33F2:23D5E5:5BCAFC:6AB89E3F